In [1]:
import os
base = '/kaggle/input/datasets/shaunthesheep/microsoft-catsvsdogs-dataset'
print(os.listdir(base))

['PetImages', 'readme[1].txt', 'MSR-LA - 3467.docx']


In [2]:
import os
import shutil
from PIL import Image, UnidentifiedImageError

def prepare_kaggle_catsdogs(
    src_dir="/kaggle/input/datasets/shaunthesheep/microsoft-catsvsdogs-dataset/PetImages",
    out_dir="/kaggle/working/data",
    val_split=0.2,
):
    # Purana incomplete folder hai to clean karo
    if os.path.exists(out_dir):
        shutil.rmtree(out_dir)

    for split in ["train", "val"]:
        for cls in ["Cat", "Dog"]:
            os.makedirs(f"{out_dir}/{split}/{cls}", exist_ok=True)

    for cls in ["Cat", "Dog"]:
        cls_dir = os.path.join(src_dir, cls)
        files = os.listdir(cls_dir)
        print(f"\n{cls}: {len(files)} total files found, checking...")

        valid_files = []
        skipped = 0
        for i, fname in enumerate(files):
            fpath = os.path.join(cls_dir, fname)
            try:
                if os.path.getsize(fpath) == 0:
                    skipped += 1
                    continue
                with Image.open(fpath) as img:
                    img.verify()
                valid_files.append(fname)
            except (UnidentifiedImageError, OSError):
                skipped += 1

            if (i + 1) % 2000 == 0:
                print(f"  checked {i+1}/{len(files)}...")

        print(f"{cls}: {len(valid_files)} valid, {skipped} skipped (corrupt)")

        split_idx = int(len(valid_files) * (1 - val_split))
        train_files = valid_files[:split_idx]
        val_files = valid_files[split_idx:]

        print(f"  linking {len(train_files)} train + {len(val_files)} val files...")
        for fname in train_files:
            src = os.path.abspath(os.path.join(cls_dir, fname))
            dst = f"{out_dir}/train/{cls}/{fname}"
            if not os.path.exists(dst):
                os.symlink(src, dst)
        for fname in val_files:
            src = os.path.abspath(os.path.join(cls_dir, fname))
            dst = f"{out_dir}/val/{cls}/{fname}"
            if not os.path.exists(dst):
                os.symlink(src, dst)

        print(f"  {cls} done.")

    print(f"\nData prepared at {out_dir}")
    return out_dir

data_path = prepare_kaggle_catsdogs()


Cat: 12501 total files found, checking...
  checked 2000/12501...
  checked 4000/12501...
  checked 6000/12501...
  checked 8000/12501...
  checked 10000/12501...
  checked 12000/12501...
Cat: 12499 valid, 2 skipped (corrupt)
  linking 9999 train + 2500 val files...
  Cat done.

Dog: 12501 total files found, checking...


/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


  checked 2000/12501...
  checked 4000/12501...
  checked 6000/12501...
  checked 8000/12501...
  checked 10000/12501...
  checked 12000/12501...
Dog: 12499 valid, 2 skipped (corrupt)
  linking 9999 train + 2500 val files...
  Dog done.

Data prepared at /kaggle/working/data


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
IMG_SIZE = 128

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

In [5]:
class CatDogCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2)

        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2)

        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(2)

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 16 * 16, 128)
        self.relu4 = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, 1)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.pool3(self.relu3(self.conv3(x)))
        x = self.flatten(x)
        x = self.relu4(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

model = CatDogCNN().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
print("Model ready. Total params:", sum(p.numel() for p in model.parameters()))

Model ready. Total params: 2120993


In [6]:
train_dataset = datasets.ImageFolder(f"{data_path}/train", transform=train_transform)
val_dataset = datasets.ImageFolder(f"{data_path}/val", transform=val_transform)

print("Classes:", train_dataset.classes)
print("Train size:", len(train_dataset), "| Val size:", len(val_dataset))

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

Classes: ['Cat', 'Dog']
Train size: 19998 | Val size: 5000


In [7]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = (torch.sigmoid(outputs) > 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total * 100


def validate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            preds = (torch.sigmoid(outputs) > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total * 100

print("Train/validate functions ready.")

Train/validate functions ready.


In [8]:
import csv
import time
from tqdm import tqdm

log_path = "/kaggle/working/training_log.csv"
best_val_acc = 0.0

with open(log_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["epoch", "train_loss", "train_acc", "val_loss", "val_acc", "epoch_time_sec"])

epochs = 10
epoch_times = []

for epoch in range(epochs):
    epoch_start = time.time()

    # ---- Training with progress bar ----
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [train]", leave=False)

    for images, labels in train_bar:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = (torch.sigmoid(outputs) > 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        train_bar.set_postfix(loss=f"{loss.item():.4f}")

    train_loss = total_loss / total
    train_acc = correct / total * 100

    # ---- Validation with progress bar ----
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [val]", leave=False)

    with torch.no_grad():
        for images, labels in val_bar:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            preds = (torch.sigmoid(outputs) > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_loss = total_loss / total
    val_acc = correct / total * 100

    # ---- Timing + ETA ----
    epoch_time = time.time() - epoch_start
    epoch_times.append(epoch_time)
    avg_epoch_time = sum(epoch_times) / len(epoch_times)
    remaining_epochs = epochs - (epoch + 1)
    eta_sec = avg_epoch_time * remaining_epochs
    eta_min = eta_sec / 60

    print(f"Epoch {epoch+1}/{epochs} | "
          f"Train loss: {train_loss:.4f}, acc: {train_acc:.2f}% | "
          f"Val loss: {val_loss:.4f}, acc: {val_acc:.2f}% | "
          f"Time: {epoch_time:.1f}s | ETA: {eta_min:.1f} min remaining")

    # ---- Save log ----
    with open(log_path, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([epoch + 1, train_loss, train_acc, val_loss, val_acc, round(epoch_time, 2)])

    # ---- Checkpoints ----
    torch.save({
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "val_acc": val_acc,
    }, "/kaggle/working/checkpoint_latest.pt")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "/kaggle/working/cat_dog_cnn_best.pt")
        print(f"  -> New best model saved (val_acc: {val_acc:.2f}%)")

total_time_min = sum(epoch_times) / 60
print(f"\nTraining complete in {total_time_min:.1f} minutes.")
print(f"Best val accuracy: {best_val_acc:.2f}%")
print(f"Log saved at: {log_path}")

Epoch 1/10 [train]:  28%|██▊       | 176/625 [00:23<00:53,  8.32it/s, loss=0.7205]/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Epoch 1/10 | Train loss: 0.5986, acc: 66.96% | Val loss: 0.5137, acc: 74.06% | Time: 96.4s | ETA: 14.5 min remaining
  -> New best model saved (val_acc: 74.06%)


Epoch 2/10 | Train loss: 0.5030, acc: 75.57% | Val loss: 0.4706, acc: 76.72% | Time: 96.6s | ETA: 12.9 min remaining
  -> New best model saved (val_acc: 76.72%)


Epoch 3/10 | Train loss: 0.4529, acc: 78.85% | Val loss: 0.4516, acc: 78.06% | Time: 96.4s | ETA: 11.3 min remaining
  -> New best model saved (val_acc: 78.06%)


Epoch 4/10 | Train loss: 0.4067, acc: 81.49% | Val loss: 0.4103, acc: 81.00% | Time: 95.8s | ETA: 9.6 min remaining
  -> New best model saved (val_acc: 81.00%)


...

Epoch 6/10 | Train loss: 0.3415, acc: 85.33% | Val loss: 0.3562, acc: 84.12% | Time: 90.4s | ETA: 6.3 min remaining
  -> New best model saved (val_acc: 84.12%)


Epoch 7/10 | Train loss: 0.3109, acc: 86.63% | Val loss: 0.3297, acc: 85.66% | Time: 90.8s | ETA: 4.7 min remaining
  -> New best model saved (val_acc: 85.66%)


Epoch 8/10 | Train loss: 0.2913, acc: 87.48% | Val loss: 0.3220, acc: 86.16% | Time: 91.1s | ETA: 3.1 min remaining
  -> New best model saved (val_acc: 86.16%)


Epoch 9/10 | Train loss: 0.2733, acc: 88.46% | Val loss: 0.3093, acc: 86.58% | Time: 91.8s | ETA: 1.6 min remaining
  -> New best model saved (val_acc: 86.58%)


Epoch 10/10 | Train loss: 0.2491, acc: 89.61% | Val loss: 0.3231, acc: 86.36% | Time: 90.8s | ETA: 0.0 min remaining

Training complete in 15.5 minutes.
Best val accuracy: 86.58%
Log saved at: /kaggle/working/training_log.csv
